In [ ]:
import os

import cv2
import numpy as np
from collections import deque

import matplotlib.pyplot as plt

In [ ]:
def remove_background(img, tol=15, fill_color=(0, 0, 0)):
    h, w = img.shape[:2]
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)
    result = img.copy()
    seeds = [(0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1)]
    for seed in seeds:
        cv2.floodFill(result, flood_mask, seed, fill_color,
                       loDiff=(tol, tol, tol), upDiff=(tol, tol, tol),
                       flags=4 | cv2.FLOODFILL_FIXED_RANGE)
    bg_mask = np.all(result == fill_color, axis=-1)
    return result, bg_mask
 
 
def preprocess_image(img, target_width=None, target_height=600,
                      remove_bg=True, bg_tol=15):
    h, w = img.shape[:2]
 
    if target_width is None and target_height is None:
        target_width, target_height = w, h
    elif target_width is not None and target_height is None:
        r = target_width / float(w)
        target_height = int(h * r)
    elif target_height is not None and target_width is None:
        r = target_height / float(h)
        target_width = int(w * r)
 
    resized = cv2.resize(img, (target_width, target_height), interpolation=cv2.INTER_LINEAR)
 
    if remove_bg:
        resized, _ = remove_background(resized, tol=bg_tol)
 
    return resized
 
 
def detect_lines_lsd(image, lsd):
    lines, widths, precisions, nfa = lsd.detect(image)
    return lines
 
 
def detect_lines_per_channel(color_img, lsd):
    b, g, r = cv2.split(color_img)
    lines_b = detect_lines_lsd(b, lsd)
    lines_g = detect_lines_lsd(g, lsd)
    lines_r = detect_lines_lsd(r, lsd)
 
    all_lines = []
    for lines in (lines_b, lines_g, lines_r):
        if lines is not None:
            all_lines.extend(lines)
    return np.array(all_lines) if all_lines else None
 
 
def find_longest_line(lines):
    longest = None
    longest_len = -1
    for line in lines:
        x1, y1, x2, y2 = np.asarray(line).reshape(-1)
        length = ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5
        if length > longest_len:
            longest_len = length
            longest = (x1, y1, x2, y2)
    return longest, longest_len
 
 
def bfs_region_color(color_img, start_x, start_y, threshold=1):
    """
    Буквальний піксельний BFS на КОЛЬОРОВОМУ зображенні.
    Сусід додається в чергу, лише якщо ЖОДЕН з каналів (B, G, R)
    не відрізняється від відповідного каналу поточного пікселя
    більш ніж на threshold.
    """
    h, w = color_img.shape[:2]
    visited = np.zeros((h, w), dtype=bool)
 
    start_x, start_y = int(round(start_x)), int(round(start_y))
    if not (0 <= start_x < w and 0 <= start_y < h):
        print(f"Стартова точка ({start_x},{start_y}) поза межами зображення")
        return visited
 
    queue = deque([(start_x, start_y)])
    visited[start_y, start_x] = True
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]
 
    img_int = color_img.astype(np.int16)  # щоб уникнути переповнення uint8 при відніманні
 
    while queue:
        cx, cy = queue.popleft()
        current_pixel = img_int[cy, cx]  # [B, G, R]
 
        for dx, dy in neighbors:
            nx, ny = cx + dx, cy + dy
            if 0 <= nx < w and 0 <= ny < h and not visited[ny, nx]:
                neighbor_pixel = img_int[ny, nx]
                diff = np.abs(neighbor_pixel - current_pixel)  # [dB, dG, dR]
 
                # Якщо ХОЧ ОДИН канал відрізняється більше ніж threshold - скіп
                if np.all(diff <= threshold):
                    visited[ny, nx] = True
                    queue.append((nx, ny))
 
    return visited
 
 
def postprocess_color(color_img, lines, output_path="postprocess_color_output.png",
                       threshold=1, seed_offset=6):
    """
    1. Знаходить найдовшу лінію серед `lines`.
    2. Від середини лінії зміщується перпендикулярно на seed_offset
       пікселів в обидва боки -> дві стартові точки.
    3. З кожної точки запускає bfs_region_color по кольоровому
       зображенню (поканальне порівняння, threshold=1).
    4. Розфарбовує знайдені регіони для візуалізації.
    """
    if lines is None or len(lines) == 0:
        raise ValueError("Немає ліній для обробки")
 
    (x1, y1, x2, y2), length = find_longest_line(lines)
    print(f"Найдовша лінія: ({x1:.1f}, {y1:.1f}) -> ({x2:.1f}, {y2:.1f}), довжина={length:.1f}")
 
    dx, dy = x2 - x1, y2 - y1
    line_len = (dx ** 2 + dy ** 2) ** 0.5
    dx, dy = dx / line_len, dy / line_len
    nx, ny = -dy, dx  # перпендикуляр до лінії
 
    mid_x, mid_y = (x1 + x2) / 2, (y1 + y2) / 2
    seed_a = (mid_x + nx * seed_offset, mid_y + ny * seed_offset)
    seed_b = (mid_x - nx * seed_offset, mid_y - ny * seed_offset)
 
    print(f"Стартова точка A: ({seed_a[0]:.1f}, {seed_a[1]:.1f})")
    print(f"Стартова точка B: ({seed_b[0]:.1f}, {seed_b[1]:.1f})")
 
    visited_a = bfs_region_color(color_img, *seed_a, threshold=threshold)
    visited_b = bfs_region_color(color_img, *seed_b, threshold=threshold)
 
    print(f"Розмір регіону A: {visited_a.sum()} пікселів")
    print(f"Розмір регіону B: {visited_b.sum()} пікселів")
 
    result = color_img.copy()
    result[visited_a] = (0, 0, 255)   # сторона A - червоний (BGR)
    result[visited_b] = (255, 0, 0)   # сторона B - синій (BGR)
 
    cv2.line(result, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
 
    cv2.imwrite(output_path, result)
    print(f"Результат збережено у: {output_path}")
 
    return visited_a, visited_b


def label_rooms(color_img, visited_a, visited_b, output_path="rooms_labeled.png",
                 min_area=300, alpha=0.45):
    """
    Малює напівпрозорий кольоровий шар для кожного значущого регіону
    і підписує кожен сегмент його ID (по центроїду).
    """
    barrier = visited_a | visited_b
    free = ~barrier

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        free.astype(np.uint8), connectivity=8
    )

    rng = np.random.RandomState(0)
    overlay = color_img.copy()
    kept_ids = []

    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] < min_area:
            continue
        kept_ids.append(i)
        color = np.array([int(c) for c in rng.randint(60, 255, size=3)])
        mask = (labels == i)
        overlay[mask] = (overlay[mask] * (1 - alpha) + color * alpha).astype(np.uint8)

    for room_id, i in enumerate(kept_ids, start=1):
        cx, cy = int(centroids[i][0]), int(centroids[i][1])
        text = str(room_id)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)
        cv2.rectangle(overlay, (cx - tw//2 - 6, cy - th//2 - 6),
                      (cx + tw//2 + 6, cy + th//2 + 6), (0, 0, 0), -1)
        cv2.putText(overlay, text, (cx - tw//2, cy + th//2),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)

    cv2.imwrite(output_path, overlay)
    return labels, stats, kept_ids

 
def process(image_path, output_path="lsd_output.png",
            postprocess_output="postprocess_color_output.png",
            threshold=2, seed_offset=6):
    img = cv2.imread(image_path)
 
    preprocessed_image = preprocess_image(img)  # кольорове, без фону
 
    lsd = cv2.createLineSegmentDetector(0)
    lines = detect_lines_per_channel(preprocessed_image, lsd)
 
    result = preprocessed_image.copy()
    if lines is not None:
        result = lsd.drawSegments(result, lines)
    cv2.imwrite(output_path, result)


    visited_a, visited_b = postprocess_color(preprocessed_image, lines, output_path=output_path,
                           threshold=threshold, seed_offset=seed_offset)
    
    labels_output_path = os.path.join(os.path.dirname(output_path), os.path.basename(output_path).split('.')[0] + "_rooms.webp")
    print(f"Labels path: {labels_output_path}")
    label_rooms(preprocessed_image, visited_a, visited_b, output_path)

In [ ]:
image_path = r"/home/petro/projects/test-assignments/room-regions-detection/data/input/image1.webp"
output_path = "/home/petro/projects/test-assignments/room-regions-detection/data/output/image1.webp"
process(image_path, output_path)

image_path = r"/home/petro/projects/test-assignments/room-regions-detection/data/input/image2.webp"
output_path = "/home/petro/projects/test-assignments/room-regions-detection/data/output/image2.webp"
process(image_path, output_path)

image_path = r"/home/petro/projects/test-assignments/room-regions-detection/data/input/image3.webp"
output_path = "/home/petro/projects/test-assignments/room-regions-detection/data/output/image3.webp"
process(image_path, output_path)